# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống (sạch) / Máy hút bụi
- `1` → Bụi

**Mở rộng:** Bổ sung thuật toán BFS và DFS để Agent tự suy nghĩ và tìm đường đến ô có bụi gần nhất, thay vì chỉ quét mù (blind search) theo hàng.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from collections import deque

In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
DUST_PROB = 0.4

# ── Tạo môi trường ──
def create_env(rows, cols, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            if random.random() < dust_prob:
                grid[r][c] = 1
    return grid

# ── Vẽ ma trận ──
def draw_grid(grid, pos, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(cols * 0.9, rows * 0.9))

    cmap = ListedColormap(['#F0F0F0', '#F4D03F'])  # 0=xám nhạt, 1=vàng
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

    for r in range(rows):
        for c in range(cols):
            if (r, c) == pos:
                ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
            elif grid[r][c] == 1:
                ax.text(c, r, '●', ha='center', va='center',
                        fontsize=16, color='#884400')
            else:
                ax.text(c, r, '0', ha='center', va='center',
                        fontsize=11, color='#888888')

    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [
        mpatches.Patch(color='#F0F0F0', label='0 - Ô sạch / Máy'),
        mpatches.Patch(color='#F4D03F', label='1 - Bụi'),
    ]
    ax.legend(handles=legend, loc='upper right',
              bbox_to_anchor=(1.35, 1.02), fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Khởi tạo ──
grid = create_env(ROWS, COLS, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu  |  Tổng bụi: {total_dust} ô")
draw_grid(grid, pos=(-1, -1), title=f'Ma trận ban đầu — Bụi: {total_dust} ô')

## 1. Thuật toán quét hàng cũ (Boustrophedon)
Giữ nguyên mã cũ để tiện so sánh.

In [ ]:
# ── Agent - Quét hàng Boustrophedon ──
def run_agent_boustrophedon(grid_in):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0

    print('=' * 50)
    print('CHẠY THUẬT TOÁN QUÉT HÀNG BOUSTROPHEDON (ORIGINAL)')
    print('=' * 50)

    # Tạo lộ trình
    path = []
    for r in range(rows):
        col_range = range(cols) if r % 2 == 0 else range(cols - 1, -1, -1)
        for c in col_range:
            path.append((r, c))

    prev = None
    for (r, c) in path:
        steps += 1

        # Xác định hướng di chuyển
        if prev is None:
            move_msg = f'Bắt đầu tại ({r}, {c})'
        else:
            pr, pc = prev
            if r > pr:
                direction = 'XUỐNG'
            elif r < pr:
                direction = 'LÊN'
            elif c > pc:
                direction = 'PHẢI'
            else:
                direction = 'TRÁI'
            move_msg = f'Bước {steps}: Di chuyển {direction} → ô ({r}, {c})'

        # Hút bụi nếu có
        if grid[r][c] == 1:
            grid[r][c] = 0
            cleaned += 1
            action_msg = f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}'
        else:
            action_msg = f'   ⟹  Ô sạch, tiếp tục.'

        print(move_msg)
        print(action_msg)
        # Tạm tắt vẽ hình ở hàm cũ để notebook đỡ dài, bạn có thể mở lại:
        # draw_grid(grid, pos=(r, c), title=f'Bước {steps} | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

        prev = (r, c)

    # Kết quả
    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Đã quét toàn bộ lưới, hút sạch hết bụi'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}\n')


## 2. Tìm đường bằng thuật toán BFS và DFS
Hàm AI để Agent suy nghĩ đường đi ngắn nhất đến ô bụi.

In [ ]:
def get_path_to_dust_bfs(grid, start_pos):
    """Duyệt theo chiều rộng (BFS) để tìm đường CẮN NHẤT đến ô có bụi"""
    rows, cols = grid.shape
    queue = deque([(start_pos, [start_pos])])
    visited = set([start_pos])
    
    while queue:
        (r, c), path = queue.popleft()
        
        # Tìm thấy bụi
        if grid[r][c] == 1:
            return path
            
        # Thử 4 hướng: Lên, Xuống, Trái, Phải
        for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [(nr, nc)]))
    return []

def get_path_to_dust_dfs(grid, start_pos):
    """Duyệt theo chiều sâu (DFS) để tìm đường đến ô có bụi"""
    rows, cols = grid.shape
    stack = [(start_pos, [start_pos])]
    visited = set([start_pos])
    
    while stack:
        (r, c), path = stack.pop()
        
        # Tìm thấy bụi
        if grid[r][c] == 1:
            return path
            
        # Đảo ngược thứ tự đưa vào stack để thứ tự duyệt giống BFS (ưu tiên Lên, Xuống, Trái, Phải)
        for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                stack.append(((nr, nc), path + [(nr, nc)]))
    return []


## 3. Agent thực thi di chuyển theo AI

In [ ]:
def run_agent_search(grid_in, algorithm='BFS'):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0
    curr_pos = (0, 0)
    
    print('=' * 50)
    print(f'CHẠY THUẬT TOÁN {algorithm}')
    print('=' * 50)
    print(f'Bắt đầu tại {curr_pos}')
    
    if grid[curr_pos[0]][curr_pos[1]] == 1:
        grid[curr_pos[0]][curr_pos[1]] = 0
        cleaned += 1
        print(f'   ⟹  Phát hiện bụi! Hút bụi tại {curr_pos} — Đã hút: {cleaned}/{total_dust}')
        # draw_grid(grid, pos=curr_pos, title=f'Bước {steps} | Vị trí: {curr_pos} | Đã hút: {cleaned}/{total_dust}')
        
    while cleaned < total_dust:
        # 1. Tìm đường đến ô bụi tiếp theo bằng AI
        if algorithm == 'BFS':
            path = get_path_to_dust_bfs(grid, curr_pos)
        else:
            path = get_path_to_dust_dfs(grid, curr_pos)
            
        if not path:
            print("Không thể tìm thấy thêm bụi nào!")
            break
            
        # 2. Di chuyển thực tế theo lộ trình AI đã vạch ra (bỏ qua vị trí hiện tại path[0])
        for next_pos in path[1:]:
            steps += 1
            r, c = next_pos
            pr, pc = curr_pos
            if r > pr: direction = 'XUỐNG'
            elif r < pr: direction = 'LÊN'
            elif c > pc: direction = 'PHẢI'
            else: direction = 'TRÁI'
            
            curr_pos = next_pos
            move_msg = f'Bước {steps}: Di chuyển {direction} → ô ({r}, {c})'
            
            if grid[r][c] == 1:
                grid[r][c] = 0
                cleaned += 1
                action_msg = f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}'
            else:
                action_msg = f'   ⟹  Ô sạch, tiếp tục.'
                
            print(move_msg)
            print(action_msg)
            
            # Tạm tắt vẽ hình để notebook không bị quá dài, bạn có thể bỏ comment dòng dưới để thấy robot di chuyển
            # draw_grid(grid, pos=(r, c), title=f'Bước {steps} | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

    # Kết quả
    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Đã tìm và hút sạch hết bụi'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}\n')


In [ ]:
# Chạy thử cả 3 thuật toán trên cùng 1 môi trường để so sánh số bước đi
run_agent_boustrophedon(grid)
run_agent_search(grid, algorithm='BFS')
run_agent_search(grid, algorithm='DFS')
